# 1. Preprocessing and Feature Engineering

In [19]:
import pandas as pd
import numpy as np

# Only keep data from June 23rd 2025 onwards (date store acquired by new owners)
df = pd.read_csv('../data/aggregated_sales_data.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df[df['Date'] >= '2025-06-23']

In [20]:
# Determine Baseline Price based on Customer Type, then check discount percentage
conditions = [
    df['Customer Type'] == 'Retail Selling Price',
    df['Customer Type'] == 'Contractor 1',
    df['Customer Type'] == 'Contractor 2',
    df['Customer Type'] == 'Contractor 3'
]

choices = [
    df['Retail Price'],
    df['Contractor 1 Price'],
    df['Contractor 2 Price'],
    df['Contractor 3 Price']
]

baseline = np.select(conditions, choices, default=df['Retail Price'])

# Calculate Discount Percentage
df['Discount_Pct'] = ((baseline - df['Unit Price']) / baseline).round(2)
df.head(10)

,Date,Customer Name,Customer Type,Item No,Product Name,Size,Quantity,Unit Price,Total Sale Price,Standard Cost,Retail Price,Contractor 1 Price,Contractor 2 Price,Contractor 3 Price,Discount_Pct
0,2026-02-19,KL Creation,Contractor 1,3PKMP-13MM,Lint Free Micro Pro - 3 Pk 13mm,Each,1,18.86,18.86,6.60,18.860000,18.860000,18.860000,18.860000,0.0
1,2026-02-19,KL Creation,Contractor 1,5PK TPL GRN,XXL PLASTIC TRAY LINERS,Each,1,7.50,7.50,4.50,7.500000,7.500000,7.500000,7.500000,0.0
2,2026-02-19,KL Creation,Contractor 1,F5241X-001,1G F5241X AURA EG INT,Gallon,1,104.77,104.77,59.10,116.860000,104.770000,96.710000,89.660000,0.0
3,2026-02-19,KL Creation,Contractor 1,F5081X-001,1G F5081X WB CEILING PNT BASE 1,Gallon,1,73.85,73.85,42.58,86.159450,73.846687,68.721756,60.828571,-0.0
4,2026-02-19,"Grieve, Carolyn",Retail Selling Price,K5521X-001,1G K5521X REGAL SLCT MA INT,Gallon,1,96.70,96.70,49.06,96.700000,83.600000,77.560000,68.490000,0.0
5,2026-02-19,Searle Quality Construction,Contractor 1,K2001X-006,COLOUR SAMPLES BASE 1,Half Pint,1,8.04,8.04,2.99,8.040000,8.040000,8.040000,8.040000,0.0
6,2026-02-19,Cash Customer,Retail Selling Price,HB285003,"DYN. 1"" FOAM BRUSH- HIGH DENSITY",Each,2,1.00,2.00,0.30,1.000000,1.000000,1.000000,1.000000,-0.0
7,2026-02-19,Cash Customer,Retail Selling Price,BNC116642,3IN HI DENSITY FM BRUSH WDH,Each,1,1.67,1.67,0.50,1.666667,1.666667,1.666667,1.666667,-0.0
8,2026-02-19,Idrenos,Contractor 2,K6273X-004,QT K6273X BEN SG INT LTX,Quart,1,28.99,28.99,10.44,35.990000,28.990000,28.990000,28.990000,0.0
9,2026-02-19,"Greco, Vanessa",Retail Selling Price,K2001X-006,COLOUR SAMPLES BASE 1,Half Pint,1,8.04,8.04,2.99,8.040000,8.040000,8.040000,8.040000,0.0


In [21]:
# Is_Promo feature
# Logic: If Discount_Pct within Promo discount range and date range, then Is_Promo = 1, else 0
import pandas as pd
import numpy as np

# Ensure Date is datetime
df['Date'] = pd.to_datetime(df['Date'])

# Define Promo Ranges
range_1_start, range_1_end = pd.Timestamp('2025-12-05'), pd.Timestamp('2025-12-31')
range_2_start, range_2_end = pd.Timestamp('2026-02-07'), pd.Timestamp('2026-02-17')

def flag_promo_1(row):
    """Flags the December Promo (Gallons Only)"""
    if row['Customer Name'] != 'Ecommerce, Shopify':
        return 0
    if not (0.14 <= row['Discount_Pct'] <= 0.3):
        return 0
    
    current_date = row['Date']
    # 1st Promo condition: Date range + Gallon size
    if (range_1_start <= current_date <= range_1_end) and (row['Size'] == 'Gallon'):
        return 1
    return 0

def flag_promo_2(row):
    """Flags the February Promo (All Paint/Sizes)"""
    if row['Customer Name'] != 'Ecommerce, Shopify':
        return 0
    if not (0.14 <= row['Discount_Pct'] <= 0.3):
        return 0
    
    current_date = row['Date']
    # 2nd Promo condition: Date range only
    if (range_2_start <= current_date <= range_2_end):
        return 1
    return 0

# Apply the functions to create separate columns
df['Is_Promo_1'] = df.apply(flag_promo_1, axis=1)
df['Is_Promo_2'] = df.apply(flag_promo_2, axis=1)

# Combined Is_Promo for general baseline filtering
df['Is_Promo'] = ((df['Is_Promo_1'] == 1) | (df['Is_Promo_2'] == 1)).astype(int)

df.head(50)

,Date,Customer Name,Customer Type,Item No,Product Name,Size,Quantity,Unit Price,Total Sale Price,Standard Cost,Retail Price,Contractor 1 Price,Contractor 2 Price,Contractor 3 Price,Discount_Pct,Is_Promo_1,Is_Promo_2,Is_Promo
0,2026-02-19,KL Creation,Contractor 1,3PKMP-13MM,Lint Free Micro Pro - 3 Pk 13mm,Each,1,18.86,18.86,6.60,18.860000,18.860000,18.860000,18.860000,0.00,0,0,0
1,2026-02-19,KL Creation,Contractor 1,5PK TPL GRN,XXL PLASTIC TRAY LINERS,Each,1,7.50,7.50,4.50,7.500000,7.500000,7.500000,7.500000,0.00,0,0,0
2,2026-02-19,KL Creation,Contractor 1,F5241X-001,1G F5241X AURA EG INT,Gallon,1,104.77,104.77,59.10,116.860000,104.770000,96.710000,89.660000,0.00,0,0,0
3,2026-02-19,KL Creation,Contractor 1,F5081X-001,1G F5081X WB CEILING PNT BASE 1,Gallon,1,73.85,73.85,42.58,86.159450,73.846687,68.721756,60.828571,-0.00,0,0,0
4,2026-02-19,"Grieve, Carolyn",Retail Selling Price,K5521X-001,1G K5521X REGAL SLCT MA INT,Gallon,1,96.70,96.70,49.06,96.700000,83.600000,77.560000,68.490000,0.00,0,0,0
5,2026-02-19,Searle Quality Construction,Contractor 1,K2001X-006,COLOUR SAMPLES BASE 1,Half Pint,1,8.04,8.04,2.99,8.040000,8.040000,8.040000,8.040000,0.00,0,0,0
6,2026-02-19,Cash Customer,Retail Selling Price,HB285003,"DYN. 1"" FOAM BRUSH- HIGH DENSITY",Each,2,1.00,2.00,0.30,1.000000,1.000000,1.000000,1.000000,-0.00,0,0,0
7,2026-02-19,Cash Customer,Retail Selling Price,BNC116642,3IN HI DENSITY FM BRUSH WDH,Each,1,1.67,1.67,0.50,1.666667,1.666667,1.666667,1.666667,-0.00,0,0,0
8,2026-02-19,Idrenos,Contractor 2,K6273X-004,QT K6273X BEN SG INT LTX,Quart,1,28.99,28.99,10.44,35.990000,28.990000,28.990000,28.990000,0.00,0,0,0
9,2026-02-19,"Greco, Vanessa",Retail Selling Price,K2001X-006,COLOUR SAMPLES BASE 1,Half Pint,1,8.04,8.04,2.99,8.040000,8.040000,8.040000,8.040000,0.00,0,0,0


In [22]:
# Separating Date feature into Day, Month, Year, and Day of the Week

df['Date'] = pd.to_datetime(df['Date'])

idx = df.columns.get_loc('Date') + 1

date_features = [
    ('Year', df['Date'].dt.year),
    ('Month', df['Date'].dt.month),
    ('Day', df['Date'].dt.day),
    ('Day of the Week', df['Date'].dt.day_name())
]

for i, (col_name, data) in enumerate(date_features):
    if col_name in df.columns:
        df.drop(columns=[col_name], inplace=True)
    df.insert(idx + i, col_name, data)

df.head()

,Date,Year,Month,Day,Day of the Week,Customer Name,Customer Type,Item No,Product Name,Size,...,Total Sale Price,Standard Cost,Retail Price,Contractor 1 Price,Contractor 2 Price,Contractor 3 Price,Discount_Pct,Is_Promo_1,Is_Promo_2,Is_Promo
0,2026-02-19,2026,2,19,Thursday,KL Creation,Contractor 1,3PKMP-13MM,Lint Free Micro Pro - 3 Pk 13mm,Each,...,18.86,6.60,18.86000,18.860000,18.860000,18.860000,0.0,0,0,0
1,2026-02-19,2026,2,19,Thursday,KL Creation,Contractor 1,5PK TPL GRN,XXL PLASTIC TRAY LINERS,Each,...,7.50,4.50,7.50000,7.500000,7.500000,7.500000,0.0,0,0,0
2,2026-02-19,2026,2,19,Thursday,KL Creation,Contractor 1,F5241X-001,1G F5241X AURA EG INT,Gallon,...,104.77,59.10,116.86000,104.770000,96.710000,89.660000,0.0,0,0,0
3,2026-02-19,2026,2,19,Thursday,KL Creation,Contractor 1,F5081X-001,1G F5081X WB CEILING PNT BASE 1,Gallon,...,73.85,42.58,86.15945,73.846687,68.721756,60.828571,-0.0,0,0,0
4,2026-02-19,2026,2,19,Thursday,"Grieve, Carolyn",Retail Selling Price,K5521X-001,1G K5521X REGAL SLCT MA INT,Gallon,...,96.70,49.06,96.70000,83.600000,77.560000,68.490000,0.0,0,0,0


In [23]:
# Standardize Size
df['Size'] = df['Size'].replace('1 gallon', 'Gallon')
df['Size'] = df['Size'].replace('18 litre', '5 Gallon')
df['Size'] = df['Size'].replace('4 Lt', 'Gallon')

# Keep only paint items
valid_sizes = ['Gallon', '5 Gallon', 'Quart', 'Half Pint']
df = df[df['Size'].isin(valid_sizes)].copy()
df.head(10)

,Date,Year,Month,Day,Day of the Week,Customer Name,Customer Type,Item No,Product Name,Size,...,Total Sale Price,Standard Cost,Retail Price,Contractor 1 Price,Contractor 2 Price,Contractor 3 Price,Discount_Pct,Is_Promo_1,Is_Promo_2,Is_Promo
2,2026-02-19,2026,2,19,Thursday,KL Creation,Contractor 1,F5241X-001,1G F5241X AURA EG INT,Gallon,...,104.77,59.10,116.860000,104.770000,96.710000,89.660000,0.00,0,0,0
3,2026-02-19,2026,2,19,Thursday,KL Creation,Contractor 1,F5081X-001,1G F5081X WB CEILING PNT BASE 1,Gallon,...,73.85,42.58,86.159450,73.846687,68.721756,60.828571,-0.00,0,0,0
4,2026-02-19,2026,2,19,Thursday,"Grieve, Carolyn",Retail Selling Price,K5521X-001,1G K5521X REGAL SLCT MA INT,Gallon,...,96.70,49.06,96.700000,83.600000,77.560000,68.490000,0.00,0,0,0
5,2026-02-19,2026,2,19,Thursday,Searle Quality Construction,Contractor 1,K2001X-006,COLOUR SAMPLES BASE 1,Half Pint,...,8.04,2.99,8.040000,8.040000,8.040000,8.040000,0.00,0,0,0
8,2026-02-19,2026,2,19,Thursday,Idrenos,Contractor 2,K6273X-004,QT K6273X BEN SG INT LTX,Quart,...,28.99,10.44,35.990000,28.990000,28.990000,28.990000,0.00,0,0,0
9,2026-02-19,2026,2,19,Thursday,"Greco, Vanessa",Retail Selling Price,K2001X-006,COLOUR SAMPLES BASE 1,Half Pint,...,8.04,2.99,8.040000,8.040000,8.040000,8.040000,0.00,0,0,0
10,2026-02-19,2026,2,19,Thursday,"Lazar, Jasmine",Contractor 2,F5491X-001,1G F5491X REGAL SLCT EG INT,Gallon,...,155.12,49.06,96.700000,83.600000,77.560000,68.490000,0.00,0,0,0
12,2026-02-19,2026,2,19,Thursday,"Hulleman Painti, Matt",Contractor 1,HP3920.7X.1FR,1G HP3920.7X HP COMMAND WB ST TNTBL WHT,Gallon,...,108.72,64.93,118.854109,112.843239,103.854766,99.846225,0.04,0,0,0
15,2026-02-19,2026,2,19,Thursday,"Johnny, Mike",Retail Selling Price,K5521X-001,1G K5521X REGAL SLCT MA INT,Gallon,...,96.70,49.06,96.700000,83.600000,77.560000,68.490000,0.00,0,0,0
16,2026-02-19,2026,2,19,Thursday,Cash Customer,Retail Selling Price,F5491X-004,NEW REGAL EGGSHELL,Quart,...,38.99,16.60,38.990000,33.990000,33.990000,33.990000,0.00,0,0,0


In [24]:
# Save the feature-engineered dataset
output_path = "../data/feature_engineered_data.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

output_path

'../data/feature_engineered_data.csv'